<a href="https://colab.research.google.com/github/EpiPandit/Pandit_et_al_WildAlert_2.0/blob/main/notebooks/circumstances_for_admission/03_01102024_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          TrainingArguments,
                          Trainer)
from datasets import load_dataset

from pathlib import Path
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
     print("cuda is not available")

NVIDIA A100-SXM4-80GB


In [ ]:
#data_dir = Path("../data/interim/")
data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/processed")
raw_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/raw")
interim_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/interim")
ckpt = "bert-base-uncased"

In [ ]:
data_dir

PosixPath('/content/drive/MyDrive/WildAlertCOA/data/processed')

In [ ]:
data_files = {
    "train": str(data_dir/"wildalert_circumstances_data_train.parquet"),
    "val": str(data_dir/"wildalert_circumstances_data_val.parquet"),
    "test": str(data_dir/"wildalert_circumstances_data_test.parquet"),
}

ds = load_dataset("parquet", data_files=data_files)
ds.set_format("torch")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24693
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6174
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5448
    })
})

In [ ]:
labels = ['abduction_with_intent_of_rescue',
 'animal_interaction',
 'bicycle_collision',
 'born_in_captivity',
 'botanicals',
 'bow_and_arrow',
 'cat_interaction',
 'collision',
 'confiscation',
 'cooking_oil_exposure',
 'displaced_from_nest',
 'disturbed_metabolic_rest',
 'dog_interaction',
 'domestic_animal_interaction',
 'dumped',
 'electrocution',
 'entrapment',
 'entrapped_in_building',
 'entrapped_in_chimney',
 'entrapped_in_fence',
 'entrapped_in_fishing_tackle',
 'entrapped_in_litter_/_garbage',
 'entrapped_in_netting_/_string_/_wire',
 'entrapped_in_storm_drain_/_sewer',
 'entrapped_in_vehicle',
 'entrapped_in_water',
 'fire_/_smoke',
 'friendly',
 'garden_/_farm_equipment_collision',
 'gas_flare',
 'grease_exposure',
 'grounded',
 'gunshot',
 'hand_held_object_collision',
 'illness',
 'inappropriate_human_intervention',
 'maladaptation_/_failure_to_thrive',
 'mating_injury',
 'nest_/_habitat_disturbance_or_destruction',
 'non-domestic_animal_interaction',
 'non-weapon_projectile',
 'nuisance_animal',
 'orphan',
 'paint_exposure',
 'pet',
 'petrochemical_exposure',
 'physical_trauma',
 'physical_trauma_by_unknown_cause',
 'plane_collision',
 'poisoned',
 'powerline_/_wire_collision',
 'referral_/_transfer',
 'same_species_interaction',
 'solar_panel_collision',
 'stranded',
 'surrender',
 'tar_exposure',
 'toxic_exposure',
 'toxin_exposure',
 'train_collision',
 'trapped_in_glue_trap',
 'trapped_in_humane_/_cage_trap',
 'trapped_in_leghold_/_trap_/_snare',
 'tree_trimming',
 'unauthorized_or_untrained_rehabilitation',
 'undetermined',
 'vehicle_collision',
 'watercraft_collision',
 'weather_event',
 'wind_turbine_collision',
 'window_/_wall_collision']

id2label = {idx:label for idx,label in enumerate(labels)}
label2id = {label:idx for idx,label in enumerate(labels)}

In [ ]:
num_labels = len(ds["train"][0]["labels"])
tokenizer = AutoTokenizer.from_pretrained(ckpt, use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
num_labels

71

In [ ]:
sample = ds["train"][18]
sample.keys()

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])

In [ ]:
tokenizer.decode(sample["input_ids"])

'[CLS] eastern cottontail. physical trauma, cat / dog attack [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'

In [ ]:
sample["labels"]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
[id2label[idx] for idx, label in enumerate(sample['labels']) if label == 1.0]

['domestic_animal_interaction', 'physical_trauma']

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
            ckpt,
            num_labels=num_labels,
            problem_type="multi_label_classification",
            id2label=id2label,
            label2id=label2id
        )

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
batch_size = 32
metric_name = "f1"

In [ ]:
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/WildAlertCOA/WildAlertCircumstancesExp",
    eval_strategy="epoch",   # ← MUST match save_strategy
    save_strategy="epoch",
    learning_rate=1e-4,
    num_train_epochs=50,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name
)

In [ ]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

5.0.0
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch

# source: https://jesusleal.io/2021/04/21/Longformer-multilabel-classification/
def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions,
            tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds,
        labels=p.label_ids)
    return result

In [ ]:
ds["train"][0]["labels"]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
ds["train"]["input_ids"][0]

tensor([  101,  2789,  6557, 14162,  1012,  2179,  1017,  2420,  3283,  1059,
         1013, 10168,  5302, 13777,  1010, 12422,  2065,  3566,  2513,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [ ]:
outputs = model(
            input_ids=ds["train"]["input_ids"][0].unsqueeze(0),
            labels=ds["train"][0]["labels"].unsqueeze(0)
)
outputs.logits

tensor([[-0.5142,  0.0256,  0.3061,  0.1792,  0.3337,  0.7593, -0.0547,  0.0159,
          0.0867,  0.2960,  0.1114, -0.0144,  0.3812,  0.1850,  0.0772, -0.3370,
         -0.1705, -0.0743,  0.1611, -0.2427, -0.1368,  0.1143,  0.0370,  0.0579,
          0.2808,  0.4507, -0.0589,  0.0481, -0.3489,  0.4699,  0.3130,  0.0579,
         -0.2527, -0.2615, -0.1721,  0.5327, -0.2279, -0.1783,  0.2210,  0.1123,
         -0.0572, -0.4561,  0.3361, -0.1853, -0.5915,  0.5447, -0.2382,  0.8823,
          0.5144,  0.2814,  0.1824, -0.2141,  0.1048,  0.1944, -0.3057, -0.0409,
         -0.0988, -0.3235, -0.0196,  0.3965,  0.6268, -0.3479, -0.6649, -0.2520,
          0.3706, -0.0131, -0.1303,  0.3116,  0.2694, -0.2124,  0.2214]],
       grad_fn=<AddmmBackward0>)

In [ ]:
%%time

# Move model to GPU
if torch.cuda.is_available():
    model = model.to("cuda")
    print("Using GPU for training")
else:
    print("CUDA is not available, using CPU")


trainer = Trainer(
    model,
    args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

Using GPU for training
CPU times: user 13.5 ms, sys: 0 ns, total: 13.5 ms
Wall time: 19.5 ms


In [ ]:
%%time
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Roc Auc,Accuracy
1,0.095238,0.033531,0.734963,0.810966,0.570133
2,0.022604,0.018866,0.846677,0.900903,0.725138
3,0.015850,0.015659,0.871364,0.926622,0.761743
4,0.011264,0.014285,0.886410,0.936728,0.789764
5,0.009058,0.013169,0.894045,0.942476,0.804503
6,0.006703,0.013254,0.891734,0.940504,0.802073
7,0.006072,0.013275,0.898424,0.948037,0.807418
8,0.004915,0.013453,0.898958,0.950092,0.811791
9,0.004492,0.014169,0.899608,0.943854,0.813411
10,0.003501,0.014416,0.897430,0.948016,0.809524


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

CPU times: user 1h 48min 20s, sys: 55.6 s, total: 1h 49min 15s
Wall time: 1h 50min 7s


TrainOutput(global_step=38600, training_loss=0.0040860735501495665, metrics={'train_runtime': 6607.4814, 'train_samples_per_second': 186.856, 'train_steps_per_second': 5.842, 'total_flos': 8.12628291595392e+16, 'train_loss': 0.0040860735501495665, 'epoch': 50.0})

In [ ]:
%%time
trainer.evaluate()

CPU times: user 10.7 s, sys: 86 ms, total: 10.8 s
Wall time: 10.7 s


{'eval_loss': 0.020875370129942894,
 'eval_f1': 0.905425219941349,
 'eval_roc_auc': 0.9549769546665412,
 'eval_accuracy': 0.8211856171039844,
 'eval_runtime': 10.7225,
 'eval_samples_per_second': 575.797,
 'eval_steps_per_second': 17.999,
 'epoch': 50.0}

In [ ]:
sample = 'Ring-necked Pheasant. Cat/Dog Bite'

In [ ]:
enc = tokenizer(sample, return_tensors="pt")

In [ ]:
enc = {k: v.to(trainer.model.device) for k,v in enc.items()}

In [ ]:
enc

{'input_ids': tensor([[  101,  3614,  1011,  3300,  2098,  6887,  5243, 22341,  1012,  4937,
           1013,  3899,  6805,   102]], device='cuda:0'),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
outputs = trainer.model(**enc)

In [ ]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-13.0723,  -5.2737, -15.0329, -14.1457, -18.9486, -14.4328,  -9.6900,
         -14.3008, -11.7774, -12.6365, -12.2015, -20.9609, -11.5498,   5.1441,
         -13.0712, -20.1696, -15.3213, -13.0304, -17.5147, -12.0467, -12.3714,
         -14.5811, -11.1730, -20.0945, -14.5454, -13.3904, -12.9782, -14.3285,
         -14.6944, -20.0679, -17.8490, -10.7102, -15.2570, -16.6257, -12.1395,
         -12.0872, -13.3844, -22.3793, -12.3670, -11.7161, -15.2569, -17.1776,
          -9.9493, -19.2969, -13.7549, -12.5700, -10.6557, -14.9477, -14.2846,
         -15.0272, -17.9398, -16.2511, -12.9079, -19.3664, -12.4699, -12.1498,
         -17.3462, -13.9030, -13.0802, -17.3943, -14.7856, -14.6229, -18.8918,
         -16.3282, -11.1196, -12.9147, -13.5990, -14.2300, -13.9243, -15.8215,
         -12.7883]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [ ]:
import torch.nn.functional as F

In [ ]:
probs = F.sigmoid(outputs.logits.squeeze().detach().cpu())

In [ ]:
probs

tensor([2.1028e-06, 5.0985e-03, 2.9601e-07, 7.1878e-07, 5.8984e-09, 5.3943e-07,
        6.1895e-05, 6.1550e-07, 7.6760e-06, 3.2510e-06, 5.0231e-06, 7.8846e-10,
        9.6379e-06, 9.9420e-01, 2.1050e-06, 1.7396e-09, 2.2185e-07, 2.1928e-06,
        2.4743e-08, 5.8641e-06, 4.2380e-06, 4.6507e-07, 1.4049e-05, 1.8753e-09,
        4.8196e-07, 1.5297e-06, 2.3102e-06, 5.9868e-07, 4.1525e-07, 1.9259e-09,
        1.7713e-08, 2.2317e-05, 2.3657e-07, 6.0191e-08, 5.3439e-06, 5.6311e-06,
        1.5389e-06, 1.9089e-10, 4.2566e-06, 8.1611e-06, 2.3660e-07, 3.4662e-08,
        4.7761e-05, 4.1636e-09, 1.0625e-06, 3.4748e-06, 2.3564e-05, 3.2234e-07,
        6.2557e-07, 2.9770e-07, 1.6176e-08, 8.7546e-08, 2.4785e-06, 3.8841e-09,
        3.8405e-06, 5.2892e-06, 2.9286e-08, 9.1621e-07, 2.0862e-06, 2.7910e-08,
        3.7904e-07, 4.4601e-07, 6.2430e-09, 8.1053e-08, 1.4819e-05, 2.4617e-06,
        1.2418e-06, 6.6065e-07, 8.9689e-07, 1.3452e-07, 2.7933e-06])

In [ ]:
preds = (probs > 0.5).int()

In [ ]:
predicted_labels = [id2label[idx] for idx, label in enumerate(preds) if label == 1.0]

In [ ]:
predicted_labels

['domestic_animal_interaction']

In [1]:
import json

# Colab's active notebook name in the session
# Replace with your filename if running locally or standard path
import google.colab._message

# Save active changes to disk first
from google.colab import drive

# Run this Python block in a cell to sanitize the file
import nbformat

# If you know your notebook path, or use a general sanitize snippet:
!pip install -q nbformat